# TP6: Dérivée totale, VJP et différentiation automatique

**IFT3395/IFT6390 — Fondements de l'apprentissage machine**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pierrelux/mlbook/blob/main/exercises/tp6_vjp.ipynb)

Ce notebook accompagne le [Chapitre 7: Réseaux de neurones](https://pierrelux.github.io/mlbook/ch7_neural_networks), sections « Rétropropagation » et « Différentiation automatique ». La séance comprend deux volets: le ou la chargé(e) de cours dérive les règles VJP au tableau, et ce notebook fournit la vérification numérique et les exercices pratiques avec JAX.

## Objectifs

À la fin de ce TP, vous serez en mesure de:
- Interpréter la dérivée totale $Df(\mathbf{a})$ comme une **application linéaire**, pas une matrice
- Relier les dérivées partielles aux **restrictions** de $Df(\mathbf{a})$ (projections et injections)
- Appliquer la règle de la chaîne comme **composition d'applications linéaires**
- Calculer des JVP et VJP avec JAX et vérifier par différences finies
- Dériver des règles VJP à la main pour des opérations courantes
- Composer des VJP pour rétropropager un gradient à travers une chaîne de fonctions
- Comparer le coût computationnel du mode avant (JVP) et du mode arrière (VJP)

Prérequis: Ch7, sections « Rétropropagation » et « Différentiation automatique » (en particulier le tableau des règles VJP et l'implémentation minimale avec `Var`/`grad`).

---

## Partie 0: Configuration

Exécutez cette cellule pour importer les bibliothèques nécessaires. Si vous utilisez Colab, JAX est déjà installé. Localement: `pip install jax`.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# Précision 64 bits — indispensable pour les vérifications par différences finies
jax.config.update("jax_enable_x64", True)

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

print(f"JAX version: {jax.__version__}")
print(f"Plateforme: {jax.devices()[0].platform}")
print("Configuration terminée!")

---
## Partie 1: La notation de Spivak — pourquoi et comment

Avant de plonger dans les calculs, clarifions la notation utilisée dans ce TP. Elle diffère peut-être de celle de vos cours de calcul, mais elle a des avantages concrets pour la suite.

### Deux traditions de notation

La notation de Leibniz — $\frac{\partial y}{\partial x}$, $\frac{\partial f}{\partial x_j}$ — est un raccourci commode, mais elle masque un fait important: la dérivée est une *fonction* (un opérateur), pas un nombre ni un rapport. Quand on écrit $\frac{\partial f}{\partial x_1}$, on ne voit pas immédiatement que le résultat dépend du point d'évaluation, ni que la « vraie » dérivée est un objet plus riche qu'une collection de dérivées partielles.

La notation de Spivak adopte une autre perspective: $Df(\mathbf{a})$ désigne la dérivée *de la fonction $f$*, évaluée *au point $\mathbf{a}$*. Le résultat est une application linéaire — une fonction qui prend un vecteur et renvoie un vecteur. Pas d'ambiguïté sur ce qu'on dérive ni où.

### Pourquoi l'adopter ici

Trois raisons pratiques:

1. **Elle correspond à JAX.** L'appel `jax.jvp(f, (a,), (v,))` applique littéralement l'application linéaire $Df(\mathbf{a})$ au vecteur $\mathbf{v}$. La fonction `f` est le premier argument, le point $\mathbf{a}$ le deuxième, la direction $\mathbf{v}$ le troisième. Pas de « $\partial$ sortie / $\partial$ entrée » — juste: fonction, point, direction.

2. **La règle de la chaîne devient limpide.** $D(g \circ f)(\mathbf{a}) = Dg(f(\mathbf{a})) \circ Df(\mathbf{a})$: c'est une composition de fonctions. Pas de $\Sigma$, pas d'indices, pas de conventions d'appariement.

3. **Elle passe à l'échelle.** La même écriture fonctionne pour des scalaires, des vecteurs, des matrices ou des tenseurs, sans changer de convention.

### Tableau de référence

Gardez ce tableau sous la main pour le reste du TP:

| Écriture Spivak | Signification | Écriture Leibniz équivalente |
|---|---|---|
| $Df(\mathbf{a})$ | Dérivée totale de $f$ en $\mathbf{a}$ (application linéaire) | $\mathbf{J}_f(\mathbf{a})$ (matrice jacobienne) |
| $Df(\mathbf{a})(\mathbf{v})$ | Appliquer la dérivée au tangent $\mathbf{v}$ | $\mathbf{J}_f(\mathbf{a})\,\mathbf{v}$ (JVP) |
| $[Df(\mathbf{a})]^*(\mathbf{u})$ | Adjoint appliqué au cotangent $\mathbf{u}$ | $\mathbf{J}_f(\mathbf{a})^\top\mathbf{u}$ (VJP) |
| $D(g \circ f)(\mathbf{a})$ | Dérivée de la composition | $\mathbf{J}_g(f(\mathbf{a}))\,\mathbf{J}_f(\mathbf{a})$ |

### La perspective opérateur

$D$ est un *opérateur* qui transforme une fonction en une nouvelle fonction:

$$f \;\longmapsto\; Df$$

Ensuite, $Df(\mathbf{a})$ est l'application linéaire obtenue en évaluant $Df$ au point $\mathbf{a}$. Enfin, $Df(\mathbf{a})(\mathbf{v})$ est cette application linéaire appliquée au vecteur $\mathbf{v}$. Trois niveaux d'« application de fonction » — et chacun correspond à un appel JAX:

| Niveau | Notation | JAX |
|--------|----------|-----|
| Opérateur de dérivation | $f \mapsto Df$ | `jax.jacobian`, `jax.jvp`, `jax.vjp` |
| Évaluation au point $\mathbf{a}$ | $Df(\mathbf{a})$ | `jax.jacobian(f)(a)` |
| Application au vecteur $\mathbf{v}$ | $Df(\mathbf{a})(\mathbf{v})$ | `jax.jvp(f, (a,), (v,))[1]` |

---
## Partie 2: La dérivée totale comme application linéaire

En calcul à une variable, on écrit $f'(a) \in \mathbb{R}$: la dérivée est un *nombre*. En plusieurs variables, la situation est plus riche. Plutôt qu'une matrice de dérivées partielles, la notion fondamentale est celle d'**application linéaire**.

**Définition (Spivak).** Soit $f: \mathbb{R}^n \to \mathbb{R}^m$. La **dérivée totale** de $f$ en $\mathbf{a}$ est l'unique application linéaire $Df(\mathbf{a}): \mathbb{R}^n \to \mathbb{R}^m$ telle que

$$\lim_{\mathbf{h} \to \mathbf{0}} \frac{\|f(\mathbf{a} + \mathbf{h}) - f(\mathbf{a}) - Df(\mathbf{a})(\mathbf{h})\|}{\|\mathbf{h}\|} = 0$$

Trois points à retenir:

1. $Df(\mathbf{a})$ est une *fonction* — elle prend un vecteur $\mathbf{v} \in \mathbb{R}^n$ et renvoie un vecteur $Df(\mathbf{a})(\mathbf{v}) \in \mathbb{R}^m$.
2. La matrice jacobienne $\mathbf{J}_f(\mathbf{a}) \in \mathbb{R}^{m \times n}$ *représente* cette application linéaire dans la base canonique: $Df(\mathbf{a})(\mathbf{v}) = \mathbf{J}_f(\mathbf{a}) \, \mathbf{v}$.
3. L'écriture $Df(\mathbf{a})(\mathbf{v})$ est exactement ce que JAX appelle un **JVP** (*Jacobian-vector product*).

Travaillons avec un exemple concret tout au long du TP.

**Exemple fil conducteur.** Considérons $f: \mathbb{R}^2 \to \mathbb{R}^3$ définie par

$$f(x_1, x_2) = \begin{pmatrix} x_1^2 + x_2 \\ x_1 x_2 \\ \sin(x_1) \end{pmatrix}$$

évaluée au point $\mathbf{a} = (1, 2)$. La jacobienne vaut

$$\mathbf{J}_f(\mathbf{a}) = \begin{pmatrix} 2x_1 & 1 \\ x_2 & x_1 \\ \cos(x_1) & 0 \end{pmatrix}\bigg|_{\mathbf{a}=(1,2)} = \begin{pmatrix} 2 & 1 \\ 2 & 1 \\ \cos(1) & 0 \end{pmatrix}$$

In [ ]:
def f(x):
    """f: R^2 -> R^3"""
    return jnp.array([x[0]**2 + x[1], x[0] * x[1], jnp.sin(x[0])])

a = jnp.array([1.0, 2.0])
print("f(a) =", f(a))

In [ ]:
# La matrice jacobienne: la REPRÉSENTATION de Df(a) dans la base canonique
J_f = jax.jacobian(f)(a)
print("Jacobienne J_f(a):")
print(J_f)
print(f"\nTaille: {J_f.shape}  (m=3 lignes, n=2 colonnes)")

Maintenant, appliquons $Df(\mathbf{a})$ *en tant que fonction* à un vecteur tangent $\mathbf{v} = (0{,}5,\; -1)$. On peut le faire de deux façons:

1. **Produit matrice-vecteur**: $\mathbf{J}_f(\mathbf{a}) \, \mathbf{v}$ (forme la matrice, puis multiplie)
2. **`jax.jvp`**: calcule $Df(\mathbf{a})(\mathbf{v})$ directement, *sans* former la matrice

Les deux donnent le même résultat — mais `jax.jvp` est plus efficace car il ne matérialise jamais la jacobienne.

In [ ]:
v = jnp.array([0.5, -1.0])

# Méthode 1: produit matrice-vecteur (forme la jacobienne complète)
Df_a_v_matrix = J_f @ v

# Méthode 2: jax.jvp — applique Df(a) comme fonction, SANS former la matrice
_, Df_a_v_jvp = jax.jvp(f, (a,), (v,))

print("J_f(a) @ v      =", Df_a_v_matrix)
print("jax.jvp(f)(a,v) =", Df_a_v_jvp)
print("Accord:", jnp.allclose(Df_a_v_matrix, Df_a_v_jvp))

### Exercice 1: Vérification de la définition de Spivak ★

La définition exige que l'erreur d'approximation linéaire décroisse *plus vite* que $\|\mathbf{h}\|$. En posant $\mathbf{h} = t \, \mathbf{v}$ pour un $\mathbf{v}$ fixé, le rapport

$$r(t) = \frac{\|f(\mathbf{a} + t\,\mathbf{v}) - f(\mathbf{a}) - Df(\mathbf{a})(t\,\mathbf{v})\|}{\|t\,\mathbf{v}\|}$$

doit tendre vers 0 quand $t \to 0$. Plus précisément, on s'attend à $r(t) = O(t)$ pour une fonction lisse (l'erreur est dominée par le terme quadratique).

Complétez le code ci-dessous pour vérifier cette propriété numériquement, puis tracez $r(t)$ en échelle log-log.

In [ ]:
t_values = np.logspace(-1, -10, 20)
ratios = []

for t in t_values:
    h = t * v
    # ============================================
    # TODO: Calculez le numérateur (norme de l'erreur d'approximation linéaire)
    # et le dénominateur (norme de h), puis le rapport r(t).
    #
    # numerateur = jnp.linalg.norm(f(a + h) - f(a) - Df(a)(h))
    #   où Df(a)(h) peut se calculer via J_f @ h ou jax.jvp
    # denominateur = jnp.linalg.norm(h)
    # ============================================
    r_t = None  # <- Complétez
    ratios.append(r_t)

# Tracé log-log
if ratios[0] is not None:
    plt.loglog(t_values, ratios, 'o-', label='$r(t)$')
    plt.loglog(t_values, t_values, '--', alpha=0.5, label='pente 1 (référence)')
    plt.xlabel('$t$')
    plt.ylabel('$r(t)$')
    plt.title('Vérification de la définition de Spivak: $r(t)$ doit décroître comme $O(t)$')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Complétez le calcul de r(t)!")

<details>
<summary><b>Solution Exercice 1</b> (cliquez pour afficher)</summary>

```python
t_values = np.logspace(-1, -10, 20)
ratios = []

for t in t_values:
    h = t * v
    linear_approx = J_f @ h  # ou: jax.jvp(f, (a,), (h,))[1]
    numerateur = jnp.linalg.norm(f(a + h) - f(a) - linear_approx)
    denominateur = jnp.linalg.norm(h)
    ratios.append(float(numerateur / denominateur))

plt.loglog(t_values, ratios, 'o-', label='$r(t)$')
plt.loglog(t_values, t_values, '--', alpha=0.5, label='pente 1 (référence)')
plt.xlabel('$t$')
plt.ylabel('$r(t)$')
plt.title('Vérification de la définition de Spivak: $r(t)$ doit décroître comme $O(t)$')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
```

On observe que $r(t)$ suit la droite de pente 1 en échelle log-log (jusqu'à ce que les erreurs d'arrondi dominent pour de très petites valeurs de $t$). Cela confirme que le résidu est $O(t)$, donc $O(\|\mathbf{h}\|)$ — l'approximation linéaire est bien meilleure que l'erreur « brute ».
</details>

---
## Partie 3: Dérivées partielles comme restrictions

On écrit souvent $\frac{\partial f_i}{\partial x_j}(\mathbf{a})$ comme si c'était une quantité primitive. En réalité, cette dérivée partielle est une **restriction** de la dérivée totale $Df(\mathbf{a})$. Deux opérations la définissent:

- **Injection** $\iota_j: \mathbb{R} \to \mathbb{R}^n$, qui envoie un scalaire $t$ sur le vecteur $t \, \mathbf{e}_j$ (perturbation le long de l'axe $j$ uniquement)
- **Projection** $\pi_i: \mathbb{R}^m \to \mathbb{R}$, qui extrait la composante $i$: $\pi_i(\mathbf{y}) = y_i$

La dérivée partielle est alors la composition

$$\frac{\partial f_i}{\partial x_j}(\mathbf{a}) = (\pi_i \circ Df(\mathbf{a}) \circ \iota_j)(1) = \bigl(Df(\mathbf{a})(\mathbf{e}_j)\bigr)_i = \bigl(\mathbf{J}_f(\mathbf{a})\bigr)_{ij}$$

Autrement dit: on injecte un vecteur unitaire $\mathbf{e}_j$ dans la dérivée totale, puis on projette sur la composante $i$. Les entrées de la jacobienne ne sont *pas* la définition — elles en sont une conséquence.

En code, $\iota_j(1) = \mathbf{e}_j$ est simplement un vecteur canonique, et $\pi_i$ est l'indexation `[i]`.

In [ ]:
def iota(j, n):
    """Injection: iota_j(1) = e_j, le j-ième vecteur canonique de R^n."""
    return jnp.eye(n)[j]

def pi(i, y):
    """Projection: pi_i(y) = y_i, la i-ième composante."""
    return y[i]

# Démonstration: calculons df_1/dx_0 (a) via la restriction
e_0 = iota(0, 2)                              # e_0 = [1, 0]
Df_a_e0 = jax.jvp(f, (a,), (e_0,))[1]        # Df(a)(e_0): colonne 0 de J_f
partial_f1_x0 = pi(1, Df_a_e0)                # pi_1(Df(a)(e_0))

print(f"pi_1 . Df(a) . iota_0  =  {partial_f1_x0}")
print(f"J_f(a)[1, 0]           =  {J_f[1, 0]}")
print(f"Accord: {jnp.isclose(partial_f1_x0, J_f[1, 0])}")

### Exercice 2: Reconstruire la jacobienne colonne par colonne (JVP) ★

Un JVP $Df(\mathbf{a})(\mathbf{e}_j)$ donne la $j$-ième *colonne* de $\mathbf{J}_f(\mathbf{a})$: c'est le vecteur des réponses de toutes les sorties à une perturbation unitaire de l'entrée $j$.

Reconstruisez la jacobienne $\mathbf{J}_f(\mathbf{a})$ colonne par colonne en utilisant $n = 2$ appels à `jax.jvp`, un par vecteur canonique $\mathbf{e}_j$.

In [ ]:
n = 2  # dimension d'entrée
m = 3  # dimension de sortie
J_par_jvp = jnp.zeros((m, n))

for j in range(n):
    e_j = iota(j, n)
    # ============================================
    # TODO: Calculez Df(a)(e_j) via jax.jvp, puis placez le résultat
    # dans la colonne j de J_par_jvp.
    #
    # col_j = jax.jvp(f, (a,), (e_j,))[1]
    # J_par_jvp = J_par_jvp.at[:, j].set(col_j)
    # ============================================
    pass

if jnp.any(J_par_jvp != 0):
    print("J_f(a) reconstruite par JVP:")
    print(J_par_jvp)
    print(f"\nAccord avec jax.jacobian: {jnp.allclose(J_par_jvp, J_f)}")
else:
    print("Complétez la boucle!")

<details>
<summary><b>Solution Exercice 2</b> (cliquez pour afficher)</summary>

```python
J_par_jvp = jnp.zeros((m, n))

for j in range(n):
    e_j = iota(j, n)
    col_j = jax.jvp(f, (a,), (e_j,))[1]
    J_par_jvp = J_par_jvp.at[:, j].set(col_j)

print("J_f(a) reconstruite par JVP:")
print(J_par_jvp)
print(f"\nAccord avec jax.jacobian: {jnp.allclose(J_par_jvp, J_f)}")
```
</details>

### Exercice 3: Reconstruire la jacobienne ligne par ligne (VJP) ★★

Le VJP offre une vue *duale*. Rappelons que `jax.vjp(f, a)` renvoie `(f(a), vjp_fn)` où `vjp_fn(u)` calcule $\mathbf{J}_f(\mathbf{a})^\top \mathbf{u}$.

Si on choisit $\mathbf{u} = \mathbf{e}_i$ (vecteur canonique de $\mathbb{R}^m$), on obtient

$$\mathbf{J}_f(\mathbf{a})^\top \mathbf{e}_i = \text{la } i\text{-ième ligne de } \mathbf{J}_f(\mathbf{a})$$

Reconstruisez la jacobienne *ligne par ligne* en utilisant $m = 3$ appels VJP.

**Observation clé:** l'exercice 2 utilise $n$ JVP (un par entrée), celui-ci utilise $m$ VJP (un par sortie). Lequel est plus économique dépend de $n$ et $m$.

In [ ]:
J_par_vjp = jnp.zeros((m, n))

# Un seul appel à jax.vjp suffit pour obtenir vjp_fn (la passe avant est partagée)
_, vjp_fn = jax.vjp(f, a)

for i in range(m):
    e_i = iota(i, m)
    # ============================================
    # TODO: Calculez J_f(a)^T e_i via vjp_fn(e_i).
    # Le résultat est la i-ième LIGNE de J_f(a).
    #
    # row_i = vjp_fn(e_i)[0]
    # J_par_vjp = J_par_vjp.at[i, :].set(row_i)
    # ============================================
    pass

if jnp.any(J_par_vjp != 0):
    print("J_f(a) reconstruite par VJP:")
    print(J_par_vjp)
    print(f"\nAccord avec jax.jacobian: {jnp.allclose(J_par_vjp, J_f)}")
else:
    print("Complétez la boucle!")

<details>
<summary><b>Solution Exercice 3</b> (cliquez pour afficher)</summary>

```python
J_par_vjp = jnp.zeros((m, n))
_, vjp_fn = jax.vjp(f, a)

for i in range(m):
    e_i = iota(i, m)
    row_i = vjp_fn(e_i)[0]
    J_par_vjp = J_par_vjp.at[i, :].set(row_i)

print("J_f(a) reconstruite par VJP:")
print(J_par_vjp)
print(f"\nAccord avec jax.jacobian: {jnp.allclose(J_par_vjp, J_f)}")
```

**Résumé:** JVP avec $\mathbf{e}_j$ extrait la **colonne** $j$, VJP avec $\mathbf{e}_i$ extrait la **ligne** $i$. Pour $f: \mathbb{R}^n \to \mathbb{R}^m$, reconstruire la jacobienne complète coûte $n$ JVP ou $m$ VJP. Quand la sortie est scalaire ($m=1$), *un seul VJP suffit* — c'est pourquoi le mode arrière domine en apprentissage machine.
</details>

---
## Partie 4: Règle de la chaîne — composition d'applications linéaires

Dans la notation de Spivak, la règle de la chaîne s'écrit

$$D(g \circ f)(\mathbf{a}) = Dg\bigl(f(\mathbf{a})\bigr) \circ Df(\mathbf{a})$$

C'est une **composition d'applications linéaires**, pas un « produit de dérivées ». En coordonnées, cela donne le produit de matrices jacobiennes:

$$\mathbf{J}_{g \circ f}(\mathbf{a}) = \mathbf{J}_g\bigl(f(\mathbf{a})\bigr) \cdot \mathbf{J}_f(\mathbf{a})$$

Définissons une seconde fonction pour composer avec $f$:

In [ ]:
def g(y):
    """g: R^3 -> R^2"""
    return jnp.array([y[0] * y[2] + y[1]**2, jnp.exp(y[0]) - y[2]])

def h(x):
    """h = g o f: R^2 -> R^2"""
    return g(f(x))

print("f(a) =", f(a))
print("h(a) = g(f(a)) =", h(a))

### Exercice 4: Trois façons de calculer la jacobienne d'une composition ★★

Calculez $\mathbf{J}_h(\mathbf{a})$ de trois manières et vérifiez qu'elles coïncident:

1. **Directement**: `jax.jacobian(h)(a)`
2. **Produit de jacobiennes**: $\mathbf{J}_g(f(\mathbf{a})) \cdot \mathbf{J}_f(\mathbf{a})$
3. **Composition de JVP**: pour chaque $\mathbf{e}_j$, calculer $Dg(f(\mathbf{a}))\bigl(Df(\mathbf{a})(\mathbf{e}_j)\bigr)$ par deux appels imbriqués à `jax.jvp`

In [ ]:
# Méthode 1: directe
J_h_direct = jax.jacobian(h)(a)
print("Méthode 1 (directe):")
print(J_h_direct)

# ============================================
# TODO — Méthode 2: produit de jacobiennes
# J_g_at_fa = jax.jacobian(g)(f(a))   # J_g évaluée en f(a)
# J_f_at_a  = jax.jacobian(f)(a)      # J_f évaluée en a
# J_h_produit = J_g_at_fa @ J_f_at_a
# ============================================
J_h_produit = None  # <- Complétez

# ============================================
# TODO — Méthode 3: composition de JVP
# J_h_jvp = jnp.zeros((2, 2))
# for j in range(2):
#     e_j = iota(j, 2)
#     _, Df_a_ej = jax.jvp(f, (a,), (e_j,))          # Df(a)(e_j)
#     _, Dg_fa_Dfej = jax.jvp(g, (f(a),), (Df_a_ej,)) # Dg(f(a))(Df(a)(e_j))
#     J_h_jvp = J_h_jvp.at[:, j].set(Dg_fa_Dfej)
# ============================================
J_h_jvp = None  # <- Complétez

if J_h_produit is not None and J_h_jvp is not None:
    print("\nMéthode 2 (produit):")
    print(J_h_produit)
    print("\nMéthode 3 (JVP composés):")
    print(J_h_jvp)
    print(f"\n1 == 2: {jnp.allclose(J_h_direct, J_h_produit)}")
    print(f"1 == 3: {jnp.allclose(J_h_direct, J_h_jvp)}")
else:
    print("\nComplétez les méthodes 2 et 3!")

<details>
<summary><b>Solution Exercice 4</b> (cliquez pour afficher)</summary>

```python
# Méthode 2: produit de jacobiennes
J_g_at_fa = jax.jacobian(g)(f(a))
J_f_at_a  = jax.jacobian(f)(a)
J_h_produit = J_g_at_fa @ J_f_at_a

# Méthode 3: composition de JVP
J_h_jvp = jnp.zeros((2, 2))
for j in range(2):
    e_j = iota(j, 2)
    _, Df_a_ej = jax.jvp(f, (a,), (e_j,))
    _, Dg_fa_Dfej = jax.jvp(g, (f(a),), (Df_a_ej,))
    J_h_jvp = J_h_jvp.at[:, j].set(Dg_fa_Dfej)
```

La méthode 3 illustre concrètement $D(g \circ f)(\mathbf{a})(\mathbf{e}_j) = Dg(f(\mathbf{a}))(Df(\mathbf{a})(\mathbf{e}_j))$ — on applique d'abord $Df(\mathbf{a})$, puis $Dg(f(\mathbf{a}))$, exactement comme une composition de fonctions.
</details>

---
## Partie 5: Le VJP — l'adjoint de la dérivée totale

La dérivée totale $Df(\mathbf{a}): \mathbb{R}^n \to \mathbb{R}^m$ est une application linéaire. Toute application linéaire entre espaces euclidiens possède un **adjoint** (ou transposée):

$$[Df(\mathbf{a})]^*: \mathbb{R}^m \to \mathbb{R}^n$$

défini par la propriété $\langle [Df(\mathbf{a})]^*(\mathbf{u}), \mathbf{v} \rangle = \langle \mathbf{u}, Df(\mathbf{a})(\mathbf{v}) \rangle$ pour tous $\mathbf{u}, \mathbf{v}$. En coordonnées:

$$[Df(\mathbf{a})]^*(\mathbf{u}) = \mathbf{J}_f(\mathbf{a})^\top \mathbf{u}$$

C'est exactement le **VJP** (*vector-Jacobian product*). Le vecteur $\mathbf{u} \in \mathbb{R}^m$ est appelé **cotangent** — il transporte un signal de la sortie vers l'entrée.

En JAX: `jax.vjp(f, a)` renvoie `(f(a), vjp_fn)`, et `vjp_fn(u)` calcule $\mathbf{J}_f(\mathbf{a})^\top \mathbf{u}$.

In [ ]:
# Démonstration: VJP de f au point a
f_a, vjp_fn = jax.vjp(f, a)

u = jnp.array([1.0, 0.0, -0.5])

# VJP via JAX
vjp_jax = vjp_fn(u)[0]

# VJP via produit matrice-vecteur explicite
vjp_explicit = J_f.T @ u

print("VJP (JAX):      ", vjp_jax)
print("J_f(a)^T @ u:   ", vjp_explicit)
print("Accord:", jnp.allclose(vjp_jax, vjp_explicit))

### Méthode systématique pour dériver une règle VJP

Avant de passer aux exercices, voici une recette en 4 étapes applicable à toute opération $f: \mathbb{R}^n \to \mathbb{R}^m$. On cherche $[Df(\mathbf{a})]^*(\mathbf{u})$ pour un cotangent $\mathbf{u} \in \mathbb{R}^m$.

> **Recette en 4 étapes:**
>
> 1. **Écrire la jacobienne.** Calculer $(\mathbf{J}_f)_{ij} = \frac{\partial f_i}{\partial x_j}(\mathbf{a})$ — les dérivées partielles usuelles.
> 2. **Transposer.** Former $\mathbf{J}_f^\top$ (échanger lignes et colonnes).
> 3. **Multiplier par $\mathbf{u}$.** Calculer $\mathbf{J}_f^\top \mathbf{u}$, c'est-à-dire la $j$-ième composante: $\displaystyle\sum_i u_i \frac{\partial f_i}{\partial x_j}(\mathbf{a})$.
> 4. **Simplifier.** Reconnaître des opérations vectorielles (produit de Hadamard, produit matrice-vecteur, etc.) pour éviter de former la matrice complète.

L'étape 4 est celle qui rend le VJP efficace: on ne forme *jamais* la jacobienne en pratique. Les exercices suivants appliquent cette recette à trois opérations courantes.

### Exercice 5: Dériver et vérifier des règles VJP ★★

Le ou la chargé(e) de cours dérive ces règles au tableau. Votre rôle est de les **vérifier numériquement** en implémentant la formule manuelle et en comparant avec `jax.vjp`.

**(a)** Carré élément par élément: $\varphi(\mathbf{x}) = \mathbf{x}^{\odot 2}$

La jacobienne est $D\varphi(\mathbf{a}) = \operatorname{diag}(2\mathbf{a})$. La règle VJP:

$$[D\varphi(\mathbf{a})]^*(\mathbf{u}) = 2\mathbf{a} \odot \mathbf{u}$$

In [ ]:
def phi(x):
    return x ** 2

a_test = jnp.array([1.0, 2.0, 3.0])
u_test = jnp.array([0.1, -0.2, 0.5])

# ============================================
# TODO (a): Implémentez la règle VJP manuellement
# vjp_manual_a = 2 * a_test * u_test
# ============================================
vjp_manual_a = None  # <- Complétez

# Vérification avec JAX
_, vjp_phi = jax.vjp(phi, a_test)
vjp_jax_a = vjp_phi(u_test)[0]

if vjp_manual_a is not None:
    print("(a) Carré élément par élément:")
    print(f"  VJP manuelle: {vjp_manual_a}")
    print(f"  VJP JAX:      {vjp_jax_a}")
    print(f"  Accord: {jnp.allclose(vjp_manual_a, vjp_jax_a)}")
else:
    print("Complétez la VJP manuelle (a)!")

**(b)** Produit matrice-vecteur: $f(\mathbf{z}) = W\mathbf{z}$ avec $W \in \mathbb{R}^{m \times n}$ fixe.

La jacobienne par rapport à $\mathbf{z}$ est $Df(\mathbf{a}) = W$ (application linéaire constante). La règle VJP:

$$[Df(\mathbf{a})]^*(\mathbf{u}) = W^\top \mathbf{u}$$

In [ ]:
W = jnp.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])  # 3x2
z = jnp.array([0.5, -1.0])
u_b = jnp.array([1.0, 0.0, -0.5])

def matmul_z(z):
    return W @ z

# ============================================
# TODO (b): VJP manuelle du produit matrice-vecteur
# vjp_manual_b = W.T @ u_b
# ============================================
vjp_manual_b = None  # <- Complétez

# Vérification avec JAX
_, vjp_matmul = jax.vjp(matmul_z, z)
vjp_jax_b = vjp_matmul(u_b)[0]

if vjp_manual_b is not None:
    print("(b) Produit matrice-vecteur:")
    print(f"  VJP manuelle: {vjp_manual_b}")
    print(f"  VJP JAX:      {vjp_jax_b}")
    print(f"  Accord: {jnp.allclose(vjp_manual_b, vjp_jax_b)}")
else:
    print("Complétez la VJP manuelle (b)!")

**(c) ★★★** Softmax: $\sigma: \mathbb{R}^K \to \mathbb{R}^K$, $\sigma_i(\mathbf{a}) = \frac{e^{a_i}}{\sum_j e^{a_j}}$.

La jacobienne a pour entrées $\frac{\partial \sigma_i}{\partial a_j} = \sigma_i(\delta_{ij} - \sigma_j)$, ce qui donne la règle VJP:

$$[D\sigma(\mathbf{a})]^*(\mathbf{u}) = \sigma(\mathbf{a}) \odot \bigl(\mathbf{u} - \langle \sigma(\mathbf{a}), \mathbf{u} \rangle \, \mathbf{1}\bigr)$$

Ce concept est subtil. Prenez le temps de dérouler le produit $\mathbf{J}_\sigma^\top \mathbf{u}$ à la main pour voir comment cette formule émerge.

In [ ]:
def softmax(a):
    a_stable = a - jnp.max(a)
    exp_a = jnp.exp(a_stable)
    return exp_a / jnp.sum(exp_a)

a_sm = jnp.array([2.0, 1.0, 0.1])
u_sm = jnp.array([1.0, -0.5, 0.3])

# ============================================
# TODO (c): VJP manuelle du softmax
# s = softmax(a_sm)
# vjp_manual_c = s * (u_sm - jnp.dot(s, u_sm))
# ============================================
vjp_manual_c = None  # <- Complétez

# Vérification avec JAX
_, vjp_softmax = jax.vjp(softmax, a_sm)
vjp_jax_c = vjp_softmax(u_sm)[0]

if vjp_manual_c is not None:
    print("(c) Softmax:")
    print(f"  VJP manuelle: {vjp_manual_c}")
    print(f"  VJP JAX:      {vjp_jax_c}")
    print(f"  Accord: {jnp.allclose(vjp_manual_c, vjp_jax_c)}")
else:
    print("Complétez la VJP manuelle (c)!")

<details>
<summary><b>Solution Exercice 5</b> (cliquez pour afficher)</summary>

```python
# (a)
vjp_manual_a = 2 * a_test * u_test

# (b)
vjp_manual_b = W.T @ u_b

# (c)
s = softmax(a_sm)
vjp_manual_c = s * (u_sm - jnp.dot(s, u_sm))
```

**Dérivation du (c).** L'entrée $(i, j)$ de la jacobienne du softmax est $\frac{\partial \sigma_i}{\partial a_j} = \sigma_i(\delta_{ij} - \sigma_j)$. On cherche $(\mathbf{J}^\top \mathbf{u})_j = \sum_i u_i \frac{\partial \sigma_i}{\partial a_j} = \sum_i u_i \sigma_i (\delta_{ij} - \sigma_j)$. On sépare: $= u_j \sigma_j - \sigma_j \sum_i u_i \sigma_i = \sigma_j (u_j - \langle \boldsymbol{\sigma}, \mathbf{u}\rangle)$.
</details>

---
## Partie 6: Composition de VJP et coût computationnel

En prenant l'adjoint de la règle de la chaîne $D(g \circ f)(\mathbf{a}) = Dg(f(\mathbf{a})) \circ Df(\mathbf{a})$, et en utilisant $(AB)^\top = B^\top A^\top$, on obtient

$$[D(g \circ f)(\mathbf{a})]^*(\mathbf{u}) = [Df(\mathbf{a})]^*\bigl([Dg(f(\mathbf{a}))]^*(\mathbf{u})\bigr)$$

L'**ordre s'inverse**: on applique d'abord l'adjoint de la fonction *extérieure* $g$, puis celui de la fonction *intérieure* $f$. C'est la rétropropagation: le signal $\mathbf{u}$ se propage de la sortie vers l'entrée.

### Exercice 6: Composer des VJP ★★

En reprenant $h = g \circ f$ et le cotangent $\mathbf{u} = (1, -1)$ (dimension de sortie de $h$), calculez $[Dh(\mathbf{a})]^*(\mathbf{u})$ de trois façons.

In [ ]:
u_h = jnp.array([1.0, -1.0])

# Méthode 1: VJP directe de h
_, vjp_h = jax.vjp(h, a)
result_1 = vjp_h(u_h)[0]
print("Méthode 1 (VJP directe):", result_1)

# ============================================
# TODO — Méthode 2: produit matriciel explicite J_f^T (J_g^T u)
# J_g_fa = jax.jacobian(g)(f(a))
# J_f_a = jax.jacobian(f)(a)
# result_2 = J_f_a.T @ (J_g_fa.T @ u_h)
# ============================================
result_2 = None  # <- Complétez

# ============================================
# TODO — Méthode 3: composition d'appels VJP
# Passe avant
# f_a = f(a)
# _, vjp_g_fn = jax.vjp(g, f_a)    # VJP de g en f(a)
# _, vjp_f_fn = jax.vjp(f, a)      # VJP de f en a
#
# Passe arrière (ordre inversé!)
# u_mid = vjp_g_fn(u_h)[0]         # [Dg(f(a))]*(u)
# result_3 = vjp_f_fn(u_mid)[0]    # [Df(a)]* o [Dg(f(a))]*(u)
# ============================================
result_3 = None  # <- Complétez

if result_2 is not None and result_3 is not None:
    print("Méthode 2 (J^T produit):", result_2)
    print("Méthode 3 (VJP composés):", result_3)
    print(f"\n1 == 2: {jnp.allclose(result_1, result_2)}")
    print(f"1 == 3: {jnp.allclose(result_1, result_3)}")
else:
    print("\nComplétez les méthodes 2 et 3!")

<details>
<summary><b>Solution Exercice 6</b> (cliquez pour afficher)</summary>

```python
# Méthode 2
J_g_fa = jax.jacobian(g)(f(a))
J_f_a = jax.jacobian(f)(a)
result_2 = J_f_a.T @ (J_g_fa.T @ u_h)

# Méthode 3
f_a = f(a)
_, vjp_g_fn = jax.vjp(g, f_a)
_, vjp_f_fn = jax.vjp(f, a)

u_mid = vjp_g_fn(u_h)[0]       # [Dg(f(a))]*(u) — signal intermédiaire
result_3 = vjp_f_fn(u_mid)[0]  # [Df(a)]*(u_mid) — signal à l'entrée
```

Notez l'ordre inversé dans la méthode 3: on propage d'abord à travers $g$ (la dernière fonction appliquée dans la passe avant), puis à travers $f$.
</details>

### Exercice 7: Coût computationnel — mode avant vs mode arrière ★★★

Considérons une chaîne de trois couches (inspirée d'un réseau de neurones avec perte scalaire):

$$f_1: \mathbb{R}^{100} \to \mathbb{R}^{50}, \quad f_2: \mathbb{R}^{50} \to \mathbb{R}^{20}, \quad f_3: \mathbb{R}^{20} \to \mathbb{R}^1$$

La composition est $\ell = f_3 \circ f_2 \circ f_1: \mathbb{R}^{100} \to \mathbb{R}^1$.

Les jacobiennes sont: $\mathbf{J}_1 \in \mathbb{R}^{50 \times 100}$, $\mathbf{J}_2 \in \mathbb{R}^{20 \times 50}$, $\mathbf{J}_3 \in \mathbb{R}^{1 \times 20}$.

Le gradient complet est $\nabla \ell = \mathbf{J}_1^\top \mathbf{J}_2^\top \mathbf{J}_3^\top \in \mathbb{R}^{100}$ (un vecteur).

**(a)** Le coût d'un produit matrice-vecteur $A \mathbf{v}$ avec $A \in \mathbb{R}^{p \times q}$ est $p \times q$ multiplications. Comptez les multiplications pour les deux ordres d'évaluation:

| Ordre | Opérations | Multiplications |
|-------|-----------|-----------------|
| Mode avant (JVP): $\mathbf{J}_3(\mathbf{J}_2(\mathbf{J}_1 \mathbf{v}))$ | $\mathbf{J}_1 \mathbf{v}$, puis $\mathbf{J}_2(\cdot)$, puis $\mathbf{J}_3(\cdot)$ | ? + ? + ? = ? par tangent |
| Mode arrière (VJP): $\mathbf{J}_1^\top(\mathbf{J}_2^\top(\mathbf{J}_3^\top u))$ | $\mathbf{J}_3^\top u$, puis $\mathbf{J}_2^\top(\cdot)$, puis $\mathbf{J}_1^\top(\cdot)$ | ? + ? + ? = ? par cotangent |

Pour le gradient complet ($\nabla \ell \in \mathbb{R}^{100}$): le mode avant requiert $n = 100$ passes JVP (une par $\mathbf{e}_j$), le mode arrière requiert $m = 1$ passe VJP. Quel est le rapport des coûts totaux?

In [ ]:
# ============================================
# TODO (a): Complétez le décompte des multiplications
# ============================================

# Mode avant (JVP) — une passe:
# J1 @ v:      50 x 100 = ?
# J2 @ (...):  20 x 50  = ?
# J3 @ (...):  1  x 20  = ?
cout_jvp_une_passe = None  # <- Complétez (somme)
cout_jvp_total = None      # <- Complétez (x 100 passes)

# Mode arrière (VJP) — une passe:
# J3^T @ u:    20 x 1   = ?
# J2^T @ (...): 50 x 20 = ?
# J1^T @ (...): 100 x 50 = ?
cout_vjp_une_passe = None  # <- Complétez (somme)
cout_vjp_total = None      # <- Complétez (x 1 passe)

if cout_jvp_total is not None:
    print(f"Mode avant:  {cout_jvp_une_passe} mult/passe × 100 passes = {cout_jvp_total}")
    print(f"Mode arrière: {cout_vjp_une_passe} mult/passe × 1 passe   = {cout_vjp_total}")
    print(f"Rapport: {cout_jvp_total / cout_vjp_total:.0f}× — égal à n (dimension d'entrée)")
else:
    print("Complétez le décompte!")

**(b)** Vérifions avec JAX. On crée une chaîne de couches linéaires (avec tanh) et on compare le temps pour calculer le gradient de deux façons: 100 JVP (mode avant) vs un seul `jax.grad` (mode arrière).

In [ ]:
import time

key = jax.random.PRNGKey(42)
keys = jax.random.split(key, 3)
W1 = jax.random.normal(keys[0], (50, 100)) * 0.1
W2 = jax.random.normal(keys[1], (20, 50)) * 0.1
W3 = jax.random.normal(keys[2], (1, 20)) * 0.1

def chain(x):
    z1 = jnp.tanh(W1 @ x)
    z2 = jnp.tanh(W2 @ z1)
    return (W3 @ z2)[0]  # scalaire

x0 = jax.random.normal(jax.random.PRNGKey(0), (100,))

# Échauffement (compilation JIT)
_ = jax.grad(chain)(x0)
_ = jax.jvp(chain, (x0,), (jnp.ones(100),))

# Mode arrière: un seul jax.grad
t0 = time.perf_counter()
for _ in range(100):
    grad_reverse = jax.grad(chain)(x0)
t_reverse = (time.perf_counter() - t0) / 100

# Mode avant: 100 JVP pour reconstruire le gradient
t0 = time.perf_counter()
for _ in range(100):
    grad_forward = jnp.zeros(100)
    for j in range(100):
        e_j = iota(j, 100)
        _, djvp = jax.jvp(chain, (x0,), (e_j,))
        grad_forward = grad_forward.at[j].set(djvp)
t_forward = (time.perf_counter() - t0) / 100

print(f"Mode arrière (1 VJP):     {t_reverse*1000:.2f} ms")
print(f"Mode avant (100 JVP):     {t_forward*1000:.2f} ms")
print(f"Rapport:                  {t_forward/t_reverse:.1f}×")
print(f"\nGradients identiques: {jnp.allclose(grad_reverse, grad_forward, atol=1e-5)}")

**(c)** Inversons les dimensions: $f_1: \mathbb{R}^1 \to \mathbb{R}^{20}$, $f_2: \mathbb{R}^{20} \to \mathbb{R}^{50}$, $f_3: \mathbb{R}^{50} \to \mathbb{R}^{100}$. Maintenant la sortie est en dimension élevée et l'entrée est scalaire.

**Question:** Combien de JVP faut-il pour la jacobienne complète? Combien de VJP? Quel mode domine?

In [ ]:
keys_inv = jax.random.split(jax.random.PRNGKey(99), 3)
W1_inv = jax.random.normal(keys_inv[0], (20, 1)) * 0.1
W2_inv = jax.random.normal(keys_inv[1], (50, 20)) * 0.1
W3_inv = jax.random.normal(keys_inv[2], (100, 50)) * 0.1

def chain_inv(x):
    """R^1 -> R^100: sortie de grande dimension."""
    z1 = jnp.tanh(W1_inv @ x)
    z2 = jnp.tanh(W2_inv @ z1)
    return W3_inv @ z2

x0_inv = jnp.array([1.0])

# Mode avant: 1 seul JVP suffit (n=1)
_, jac_forward = jax.jvp(chain_inv, (x0_inv,), (jnp.ones(1),))

# Mode arrière: 100 VJP nécessaires (m=100)
_, vjp_inv_fn = jax.vjp(chain_inv, x0_inv)
jac_reverse = jnp.zeros((100, 1))
for i in range(100):
    e_i = iota(i, 100)
    row_i = vjp_inv_fn(e_i)[0]
    jac_reverse = jac_reverse.at[i, :].set(row_i)

print(f"Jacobienne (mode avant, 1 JVP):  forme {jac_forward.shape}")
print(f"Jacobienne (mode arrière, 100 VJP): forme {jac_reverse.shape}")
print(f"Accord: {jnp.allclose(jac_forward.reshape(-1), jac_reverse.reshape(-1), atol=1e-5)}")
print(f"\nRègle générale: n < m → mode avant gagne; n > m → mode arrière gagne.")
print(f"En apprentissage machine, la perte est scalaire (m=1): le mode arrière domine toujours.")

<details>
<summary><b>Solution Exercice 7 (a)</b> (cliquez pour afficher)</summary>

| Ordre | Opérations | Multiplications |
|-------|-----------|-----------------|
| Mode avant (JVP) | $50 \times 100 + 20 \times 50 + 1 \times 20$ | $5000 + 1000 + 20 = 6020$ par tangent |
| Mode arrière (VJP) | $20 \times 1 + 50 \times 20 + 100 \times 50$ | $20 + 1000 + 5000 = 6020$ par cotangent |

Le coût par passe est le même! La différence vient du nombre de passes:
- Mode avant: $100$ passes (une par $\mathbf{e}_j$) → $100 \times 6020 = 602\,000$
- Mode arrière: $1$ passe (un seul cotangent $u = 1$) → $1 \times 6020 = 6020$
- **Rapport: $602\,000 / 6020 = 100 = n$**, la dimension d'entrée

```python
cout_jvp_une_passe = 50*100 + 20*50 + 1*20    # = 6020
cout_jvp_total     = cout_jvp_une_passe * 100  # = 602000

cout_vjp_une_passe = 20*1 + 50*20 + 100*50    # = 6020
cout_vjp_total     = cout_vjp_une_passe * 1    # = 6020
```
</details>

---
## Partie 7: Vérification par différences finies

Les différences finies fournissent un outil de débogage indispensable. L'approximation par différences centrées:

$$\frac{\partial f_i}{\partial x_j}(\mathbf{a}) \approx \frac{f_i(\mathbf{a} + \varepsilon \, \mathbf{e}_j) - f_i(\mathbf{a} - \varepsilon \, \mathbf{e}_j)}{2\varepsilon}$$

donne une erreur $O(\varepsilon^2)$ (bien meilleure que la différence avant, qui est $O(\varepsilon)$). Le choix de $\varepsilon$ est un compromis: trop grand → erreur de troncature; trop petit → erreurs d'arrondi en virgule flottante. En pratique, $\varepsilon \approx 10^{-7}$ en `float64` est un bon choix.

### Exercice 8: Vérificateur de jacobienne par différences finies ★

Implémentez une fonction qui approxime la jacobienne par différences finies centrées, puis comparez avec la jacobienne exacte de $f$ au point $\mathbf{a}$.

In [ ]:
def finite_diff_jacobian(f, x, eps=1e-7):
    """Jacobienne par différences finies centrées."""
    x = jnp.asarray(x, dtype=jnp.float64)
    f_x = f(x)
    n = x.shape[0]
    m = f_x.shape[0]
    J = np.zeros((m, n))
    # ============================================
    # TODO: Pour chaque j de 0 à n-1:
    #   e_j = iota(j, n)
    #   J[:, j] = (f(x + eps * e_j) - f(x - eps * e_j)) / (2 * eps)
    # ============================================
    return J

# Test
J_fd = finite_diff_jacobian(f, a)
if np.any(J_fd != 0):
    print("Jacobienne par différences finies:")
    print(J_fd)
    print(f"\nJacobienne exacte (JAX):")
    print(np.array(J_f))
    print(f"\nErreur max: {np.max(np.abs(J_fd - np.array(J_f))):.2e}")
else:
    print("Complétez finite_diff_jacobian!")

<details>
<summary><b>Solution Exercice 8</b> (cliquez pour afficher)</summary>

```python
def finite_diff_jacobian(f, x, eps=1e-7):
    x = jnp.asarray(x, dtype=jnp.float64)
    f_x = f(x)
    n = x.shape[0]
    m = f_x.shape[0]
    J = np.zeros((m, n))
    for j in range(n):
        e_j = iota(j, n)
        J[:, j] = (f(x + eps * e_j) - f(x - eps * e_j)) / (2 * eps)
    return J
```
</details>

Traçons l'erreur en fonction de $\varepsilon$ pour observer le compromis troncature/arrondi.

In [ ]:
epsilons = np.logspace(-1, -15, 30)
errors = []

J_exact = np.array(jax.jacobian(f)(a))
for eps in epsilons:
    J_fd_eps = finite_diff_jacobian(f, a, eps=eps)
    errors.append(np.max(np.abs(J_fd_eps - J_exact)))

plt.loglog(epsilons, errors, 'o-')
plt.xlabel('$\\varepsilon$')
plt.ylabel('Erreur max $\\|\\mathbf{J}_{\\mathrm{fd}} - \\mathbf{J}_{\\mathrm{exact}}\\|_\\infty$')
plt.title('Compromis troncature/arrondi des différences finies centrées')
plt.axvline(1e-7, color='red', linestyle='--', alpha=0.5, label='$\\varepsilon = 10^{-7}$')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Récapitulatif

Ce TP a relié la perspective abstraite de Spivak au code JAX concret. Le tableau suivant résume les correspondances:

| Concept (Spivak) | Notation | Code JAX | Interprétation |
|---|---|---|---|
| Dérivée totale | $Df(\mathbf{a})$ | `jax.jacobian(f)(a)` (matrice) | Application linéaire $\mathbb{R}^n \to \mathbb{R}^m$ |
| Application à un tangent | $Df(\mathbf{a})(\mathbf{v})$ | `jax.jvp(f, (a,), (v,))[1]` | JVP — mode avant |
| Adjoint | $[Df(\mathbf{a})]^*(\mathbf{u})$ | `jax.vjp(f, a)[1](u)[0]` | VJP — mode arrière |
| Dérivée partielle | $\pi_i \circ Df(\mathbf{a}) \circ \iota_j$ | `jax.jacobian(f)(a)[i, j]` | Restriction de $Df(\mathbf{a})$ |
| Règle de la chaîne | $D(g \circ f)(\mathbf{a}) = Dg(f(\mathbf{a})) \circ Df(\mathbf{a})$ | Composer `jax.jvp` ou `jax.vjp` | Composition d'opérateurs linéaires |

Les points essentiels:

1. La **dérivée totale** est une application linéaire, pas une matrice. La matrice jacobienne la *représente*.
2. Les **dérivées partielles** sont des restrictions: $\frac{\partial f_i}{\partial x_j} = \pi_i \circ Df(\mathbf{a}) \circ \iota_j$.
3. La **règle de la chaîne** est une composition d'applications linéaires. En prenant l'adjoint, l'ordre s'inverse.
4. Le **VJP** est l'adjoint de la dérivée totale: $[Df(\mathbf{a})]^*(\mathbf{u}) = \mathbf{J}^\top \mathbf{u}$.
5. Pour $f: \mathbb{R}^n \to \mathbb{R}^1$ (perte scalaire), un seul VJP donne le gradient complet, contre $n$ JVP. C'est l'avantage fondamental du **mode arrière** (rétropropagation).

---

**Pour aller plus loin**: [Chapitre 7: Réseaux de neurones](https://pierrelux.github.io/mlbook/ch7_neural_networks), sections « Règles VJP » et « Implémentation minimale ».